# CS336 Triton T4 Lab
Stanford CS336 Spring 2026 Lecture 6 + Assignment 2 + Lecture 5의 GPU kernel 실습을 T4/Colab에서 반복한다.

반복 구조: **PyTorch baseline → benchmark/profile → Triton → correctness → benchmark/profile → PTX → 병목/개선**

항목: GeLU / Softmax / Row Sum / MatMul+ReLU / RMSNorm / Attention(online softmax forward).

In [ ]:
import sys, subprocess, importlib.util, math, torch, torch.nn.functional as F
if importlib.util.find_spec("triton") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","triton"])
import triton, triton.language as tl
from torch.profiler import profile, ProfilerActivity
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0), torch.__version__, triton.__version__)

def bench(fn,warmup=5,trials=20):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    ts=[]
    for _ in range(trials):
        a,b=torch.cuda.Event(True),torch.cuda.Event(True)
        a.record(); fn(); b.record(); torch.cuda.synchronize()
        ts.append(a.elapsed_time(b))
    return sum(ts)/len(ts)

def prof(fn):
    for _ in range(2): fn()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CUDA]) as p:
        fn(); torch.cuda.synchronize()
    print(p.key_averages().table(sort_by="cuda_time_total",row_limit=10,max_name_column_width=90))

def ptx(k,n=25):
    lines=[x for x in k.asm.get("ptx","").splitlines() if "global" in x or "%ctaid" in x or "%tid" in x]
    print("\n".join(lines[:n]))

def show(**fs):
    for k,v in fs.items(): print(f"{k:>18}: {bench(v):.4f} ms")

# 1. GeLU — elementwise + fusion
Stanford Lecture 6 첫 Triton 예제. naive/builtin/`torch.compile`/Triton의 kernel 수와 HBM 왕복을 비교한다.

In [ ]:
def naive_gelu(x):
    a=.79788456*(x+.044715*x*x*x)
    return .5*x*(1+torch.tanh(a))
compiled_gelu=torch.compile(naive_gelu)

@triton.jit
def gelu_k(x,y,n:tl.constexpr,B:tl.constexpr):
    o=tl.program_id(0)*B+tl.arange(0,B); m=o<n
    z=tl.load(x+o,mask=m); a=.79788456*(z+.044715*z*z*z)
    e=tl.exp(2*a); z=.5*z*(1+(e-1)/(e+1))
    tl.store(y+o,z,mask=m)
def tgelu(x,ret=False):
    y=torch.empty_like(x); B=1024
    k=gelu_k[(triton.cdiv(x.numel(),B),)](x,y,x.numel(),B=B)
    return (y,k) if ret else y

x=torch.randn(1<<20,device="cuda")
y,k=tgelu(x,True); torch.testing.assert_close(y,naive_gelu(x),atol=2e-3,rtol=2e-3)
show(naive=lambda:naive_gelu(x),builtin=lambda:F.gelu(x,approximate="tanh"),compiled=lambda:compiled_gelu(x),triton=lambda:tgelu(x))
print("naive"); prof(lambda:naive_gelu(x))
print("triton"); prof(lambda:tgelu(x))
ptx(k)

# 2. Softmax — reduction + fusion
`max → sub → exp → sum → div`를 분리한 PyTorch와 한 row를 한 Triton program에서 처리하는 fused kernel을 비교한다.

In [ ]:
def nsoft(x):
    z=x-x.max(1,keepdim=True).values; e=z.exp()
    return e/e.sum(1,keepdim=True)

@triton.jit
def soft_k(x,y,N:tl.constexpr,B:tl.constexpr):
    r=tl.program_id(0); o=tl.arange(0,B); m=o<N
    z=tl.load(x+r*N+o,mask=m,other=-float("inf"))
    z=z-tl.max(z,axis=0); e=tl.exp(z); z=e/tl.sum(e,axis=0)
    tl.store(y+r*N+o,z,mask=m)
def tsoft(x,ret=False):
    R,N=x.shape; B=triton.next_power_of_2(N); y=torch.empty_like(x)
    k=soft_k[(R,)](x,y,N=N,B=B)
    return (y,k) if ret else y

sx=torch.randn(4096,1024,device="cuda")
y,k=tsoft(sx,True); torch.testing.assert_close(y,torch.softmax(sx,1),atol=2e-5,rtol=2e-5)
show(naive=lambda:nsoft(sx),pytorch=lambda:torch.softmax(sx,1),triton=lambda:tsoft(sx))
prof(lambda:nsoft(sx)); prof(lambda:tsoft(sx)); ptx(k)

# 3. Row Sum — baby tiling
Stanford의 'row가 한 block에 안 들어가는 reduction' 예제. tile 크기를 바꾸며 큰 row를 순회한다.

In [ ]:
@triton.jit
def rowsum_k(x,y,N:tl.constexpr,B:tl.constexpr):
    r=tl.program_id(0); acc=tl.zeros((B,),tl.float32)
    for s in range(0,N,B):
        o=s+tl.arange(0,B); acc+=tl.load(x+r*N+o,mask=o<N,other=0.)
    tl.store(y+r,tl.sum(acc,axis=0))
def rowsum(x,B=1024,ret=False):
    R,N=x.shape; y=torch.empty(R,device=x.device)
    k=rowsum_k[(R,)](x,y,N=N,B=B)
    return (y,k) if ret else y

rx=torch.randn(2048,8192,device="cuda")
y,k=rowsum(rx,1024,True); torch.testing.assert_close(y,rx.sum(1),atol=2e-2,rtol=2e-4)
for B in [256,512,1024,2048]: print(B,bench(lambda B=B:rowsum(rx,B)))
print("torch",bench(lambda:rx.sum(1))); prof(lambda:rowsum(rx)); ptx(k)

# 4. MatMul + ReLU — tiling + epilogue fusion
Stanford Lecture 6 네 번째 Triton 예제. `torch.matmul`이 더 빠를 수 있으며 목적은 tile 재사용과 ReLU fusion 확인이다.

In [ ]:
@triton.jit
def mm_k(a,b,c,M:tl.constexpr,N:tl.constexpr,K:tl.constexpr,BM:tl.constexpr,BN:tl.constexpr,BK:tl.constexpr):
    pm,pn=tl.program_id(0),tl.program_id(1)
    om=pm*BM+tl.arange(0,BM); on=pn*BN+tl.arange(0,BN); ok=tl.arange(0,BK)
    acc=tl.zeros((BM,BN),tl.float32)
    for s in range(0,K,BK):
        A=tl.load(a+om[:,None]*K+(s+ok[None,:]),mask=(om[:,None]<M)&(s+ok[None,:]<K),other=0.)
        B=tl.load(b+(s+ok[:,None])*N+on[None,:],mask=(s+ok[:,None]<K)&(on[None,:]<N),other=0.)
        acc+=tl.dot(A,B)
    acc=tl.maximum(acc,0.)
    tl.store(c+om[:,None]*N+on[None,:],acc,mask=(om[:,None]<M)&(on[None,:]<N))
def tmm(a,b,BM=32,BN=32,BK=32,ret=False):
    M,K=a.shape; N=b.shape[1]; c=torch.empty((M,N),device=a.device,dtype=a.dtype)
    k=mm_k[(triton.cdiv(M,BM),triton.cdiv(N,BN))](a,b,c,M=M,N=N,K=K,BM=BM,BN=BN,BK=BK,num_warps=4)
    return (c,k) if ret else c

a=torch.randn(1024,1024,device="cuda",dtype=torch.float16); b=torch.randn_like(a)
y,k=tmm(a,b,ret=True); torch.testing.assert_close(y,torch.relu(a@b),atol=.2,rtol=.02)
show(torch=lambda:torch.relu(a@b),triton=lambda:tmm(a,b))
for t in [(16,16,32),(32,32,32),(64,32,32)]: print(t,bench(lambda t=t:tmm(a,b,*t)))
prof(lambda:torch.relu(a@b)); prof(lambda:tmm(a,b)); ptx(k)

# 5. RMSNorm — Assignment 2 fused kernel
Assignment 2의 대표 kernel. reduction + normalize + scale을 한 program에서 처리해 memory-bound fusion을 관찰한다.

In [ ]:
def rms(x,w,e=1e-6):
    r=torch.rsqrt(x.float().square().mean(-1,keepdim=True)+e)
    return (x.float()*r*w.float()).to(x.dtype)
crms=torch.compile(rms)

@triton.jit
def rms_k(x,w,y,N:tl.constexpr,e:tl.constexpr,B:tl.constexpr):
    r=tl.program_id(0); o=tl.arange(0,B); m=o<N
    z=tl.load(x+r*N+o,mask=m,other=0.).to(tl.float32)
    ww=tl.load(w+o,mask=m,other=0.).to(tl.float32)
    inv=tl.rsqrt(tl.sum(z*z,axis=0)/N+e)
    tl.store(y+r*N+o,z*inv*ww,mask=m)
def trms(x,w,e=1e-6,ret=False):
    R,N=x.shape; B=triton.next_power_of_2(N); y=torch.empty_like(x)
    k=rms_k[(R,)](x,w,y,N=N,e=e,B=B)
    return (y,k) if ret else y

nx=torch.randn(4096,4096,device="cuda",dtype=torch.float16); nw=torch.randn(4096,device="cuda",dtype=torch.float16)
y,k=trms(nx,nw,ret=True); torch.testing.assert_close(y,rms(nx,nw),atol=.03,rtol=.03)
show(pytorch=lambda:rms(nx,nw),compiled=lambda:crms(nx,nw),triton=lambda:trms(nx,nw))
prof(lambda:rms(nx,nw)); prof(lambda:trms(nx,nw)); ptx(k)

# 6. Attention — tiled online-softmax full forward
Lecture 5 FlashAttention의 핵심을 block 하나 수준에서 직접 구현한다. forward-only/non-causal, `D=64` 중심.

In [ ]:
def natt(q,k,v):
    s=q.shape[-1]**-.5; p=torch.softmax((q@k.transpose(-2,-1))*s,-1); return p@v
def sdpa(q,k,v): return F.scaled_dot_product_attention(q,k,v)

@triton.jit
def attn_k(q,k,v,o,S:tl.constexpr,D:tl.constexpr,scale:tl.constexpr,BM:tl.constexpr,BN:tl.constexpr):
    pm,bh=tl.program_id(0),tl.program_id(1); base=bh*S*D
    om=pm*BM+tl.arange(0,BM); on=tl.arange(0,BN); od=tl.arange(0,D)
    Q=tl.load(q+base+om[:,None]*D+od[None,:],mask=om[:,None]<S,other=0.)
    mi=tl.full((BM,),-float("inf"),tl.float32); li=tl.zeros((BM,),tl.float32); acc=tl.zeros((BM,D),tl.float32)
    for s in range(0,S,BN):
        n=s+on
        K=tl.load(k+base+n[:,None]*D+od[None,:],mask=n[:,None]<S,other=0.)
        V=tl.load(v+base+n[:,None]*D+od[None,:],mask=n[:,None]<S,other=0.)
        qk=tl.dot(Q,tl.trans(K))*scale; qk=tl.where(n[None,:]<S,qk,-float("inf"))
        mij=tl.max(qk,axis=1); mn=tl.maximum(mi,mij); alpha=tl.exp(mi-mn); P=tl.exp(qk-mn[:,None])
        li=li*alpha+tl.sum(P,axis=1); acc=acc*alpha[:,None]+tl.dot(P.to(V.dtype),V); mi=mn
    tl.store(o+base+om[:,None]*D+od[None,:],acc/li[:,None],mask=om[:,None]<S)
def tatt(q,k,v,ret=False):
    B,H,S,D=q.shape; o=torch.empty_like(q); BM=BN=32
    kk=attn_k[(triton.cdiv(S,BM),B*H)](q,k,v,o,S=S,D=D,scale=D**-.5,BM=BM,BN=BN,num_warps=4)
    return (o,kk) if ret else o

q=torch.randn(2,8,512,64,device="cuda",dtype=torch.float16); k=torch.randn_like(q); v=torch.randn_like(q)
y,kk=tatt(q,k,v,True); torch.testing.assert_close(y,sdpa(q,k,v),atol=.06,rtol=.06)
show(naive=lambda:natt(q,k,v),sdpa=lambda:sdpa(q,k,v),triton=lambda:tatt(q,k,v))
prof(lambda:natt(q,k,v)); prof(lambda:tatt(q,k,v)); ptx(kk)
for S in [512,1024,2048,4096]:
    print(S, f"score tensor ≈ {2*2*8*S*S/1024**2:.1f} MiB (fp16, one score matrix)")

# 반복 체크
각 절에서 확인:
1. kernel이 몇 개 뜨는가?
2. 중간 tensor가 HBM에 materialize되는가?
3. fusion이 어떤 round-trip을 없앴는가?
4. tiling이 어떤 데이터를 재사용하는가?
5. PTX의 global load/store가 어떻게 보이는가?
6. T4에서 병목이 launch/memory/compute 중 어디에 가까운가?

**Source mapping**
- `stanford-cs336/lectures/lecture_06.py`: benchmark/profile, GeLU, softmax, row sum, matmul+ReLU, PTX
- `stanford-cs336/assignment2-systems`: fused RMSNorm
- CS336 Lecture 5: FlashAttention / online softmax / data movement